In [14]:
import polars as pl

df = pl.scan_parquet("/Users/emiliodulay/Documents/1. UCLA/2. Year 2/3. Spring 2026/STAT M148/statM148proj/training_labeled_flattened.parquet")
print(df.head(1).collect())

shape: (1, 3)
┌──────────────────────┬─────────────────────────────────┬────────────┐
│ id                   ┆ journey                         ┆ is_success │
│ ---                  ┆ ---                             ┆ ---        │
│ str                  ┆ list[struct[2]]                 ┆ bool       │
╞══════════════════════╪═════════════════════════════════╪════════════╡
│ -635794259 449271353 ┆ [{2021-10-19 06:00:00 UTC,2}, … ┆ false      │
└──────────────────────┴─────────────────────────────────┴────────────┘


In [15]:
# Convert bool label → int  (True → 1, False → 0)
df = df.with_columns(
    pl.col("is_success").cast(pl.Int64).alias("label")
)

In [16]:
# 1. Grab both columns and bring them into memory once
batch = df.select(["id", "label"]).collect()

# 2. Extract the Series and zip them
# In Polars, df["col_name"] returns a Series, which works perfectly in zip()
label_dict = dict(zip(batch["id"], batch["label"]))


# Split by id (never split mid-journey)
ids = batch["id"].to_list() # Creates a list of ids
split = int(len(ids) * 0.8) # 80% train, 20% val
train_ids, val_ids = set(ids[:split]), set(ids[split:]) # Creates a set of train ids and a set of validation ids

train_df = df.filter(pl.col("id").is_in(train_ids)).select(["id", "journey"]) # gets the ids and journeys corresponding to the train_ids
val_df   = df.filter(pl.col("id").is_in(val_ids)).select(["id", "journey"]) # get the ids and journeys corresponding to the val_ids

train_labels = {k: v for k, v in label_dict.items() if k in train_ids}
val_labels   = {k: v for k, v in label_dict.items() if k in val_ids}


from emilio_dataset import prepare_datasets
# Pass into the pipeline from dataset.py
train_loader, val_loader, _, scaler = prepare_datasets(
    train_df, val_df, val_df,   # using val as test placeholder for now
    train_labels, val_labels, val_labels,
    max_seq_len=256,
    batch_size=32,
)

# Build the model — binary classification = 2 classes
from time_aware_transformer import make_classification_transformer

model = make_classification_transformer(
    num_features=8,    # however many features you extract per event
    num_classes=2,     # successful / not successful
)

## Full training Loop

In [17]:
# Grab one batch
example_batch = next(iter(train_loader))
print(f"Source shape: {example_batch['src'].shape}") 
# Should output: torch.Size([32, 256, 8])

Source shape: torch.Size([32, 256, 8])


In [18]:
import torch
import torch.nn as nn

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
criterion = nn.CrossEntropyLoss()

for epoch in range(1):
    model.train()
    for batch in train_loader:
        src   = batch["src"].to(device)           # (B, T, num_features)
        mask  = batch["padding_mask"].to(device)  # (B, T)
        label = batch["label"].to(device)         # (B,)  — 0 or 1

        logits = model(src, src_padding_mask=mask) # (B, 2)
        loss   = criterion(logits, label)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    print(f"START OF VALIDATION FOR EPOCH {epoch+1}")

    # Validation
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            logits = model(
                batch["src"].to(device),
                src_padding_mask=batch["padding_mask"].to(device)
            )
            preds   = logits.argmax(dim=-1)       # (B,)
            correct += (preds == batch["label"].to(device)).sum().item()
            total   += len(preds)
    print(f"Epoch {epoch+1}  val accuracy: {correct/total:.3f}")

TypeError: TimeSeriesTransformer.forward() missing 1 required positional argument: 'timestamps'

# Testing Set Predictions

In [ ]:
test_df = pl.scan_parquet("/Users/emiliodulay/Documents/1. UCLA/2. Year 2/3. Spring 2026/STAT M148/statM148proj/test_labeled_flattened.parquet")

In [ ]:
import torch
import polars as pl
from emilio_dataset import prepare_datasets

# ── 1. Test set has no labels — create dummy labels so prepare_datasets doesn't break ──
test_collected = test_df.select(["id", "journey"]).collect()
test_ids_list  = test_collected["id"].to_list()
dummy_labels   = {id_: 0 for id_ in test_ids_list}   # placeholder, never used

test_seq_df = test_df.select(["id", "journey"])

# ── 2. Create test DataLoader ──
_, _, test_loader, _ = prepare_datasets(
    train_df, val_df, test_seq_df,
    train_labels, val_labels, dummy_labels,
    max_seq_len=256,
    batch_size=32,
)

# ── 3. Run inference ──
model.eval()

all_ids, all_preds, all_probs = [], [], []

with torch.no_grad():
    for batch in test_loader:
        logits = model(
            batch["src"].to(device),
            src_padding_mask=batch["padding_mask"].to(device)
        )
        probs = torch.softmax(logits, dim=-1)
        preds = logits.argmax(dim=-1)

        all_ids.extend(batch["id"])
        all_preds.extend(preds.cpu().tolist())
        all_probs.extend(probs[:, 1].cpu().tolist())  # prob of class 1 (success)

# ── 4. Save submission file ──
results = pl.DataFrame({
    "id":           all_ids,
    "predicted":    all_preds,
    "prob_success": all_probs,
})

submission = pl.DataFrame({
    "id": all_ids,
    "predictied" : all_preds
})

submission.write_csv("submission.csv")
print(submission.head(10))
print(f"\n{len(submission)} predictions saved to submission.csv")

shape: (10, 2)
┌─────────────────────────┬────────────┐
│ id                      ┆ predictied │
│ ---                     ┆ ---        │
│ str                     ┆ i64        │
╞═════════════════════════╪════════════╡
│ -1000001271 551641434   ┆ 0          │
│ -100001164 -1710062169  ┆ 0          │
│ -1000073039 494887319   ┆ 0          │
│ -1000092799 -1963858498 ┆ 0          │
│ -100009516 1394046265   ┆ 0          │
│ -1000106108 -1431397518 ┆ 0          │
│ -1000116545 -297852619  ┆ 0          │
│ -1000129097 -1706610916 ┆ 0          │
│ -1000140359 1713232320  ┆ 0          │
│ -1000176032 1058230727  ┆ 0          │
└─────────────────────────┴────────────┘

158325 predictions saved to submission.csv


test_df.head().collect()